In [1]:
import os
import io
from PIL import Image

# BUGFIX: The current environment override can trigger the known protobuf/TensorFlow
# incompatibility on Kaggle Py3.12 (MessageFactory.GetPrototype). Let Kaggle defaults work.
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", None)
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", None)

os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"

TF_AVAILABLE = True
try:
    import tensorflow as tf
except Exception as e:
    TF_AVAILABLE = False
    tf = None
    print(
        "WARNING: TensorFlow import failed; will write fallback submission. Error:",
        repr(e),
    )

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("TensorFlow available =", TF_AVAILABLE)
if TF_AVAILABLE:
    print("TensorFlow version =", tf.__version__)

PLATFORM = "kaggle"  # local kaggle
NEEDTRAIN = False
DATATYPE = ["eeg", "spe", "img"]  # 'eeg', 'spe', 'img', 'stft'
STAGETRAIN = [2, 3]
STAGETEST = 3
print(DATATYPE)

LOAD_MODELS_FROM = "models2024030301"
if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"

EEG_LENGTH = 30  # s
SFREQ = 100

HIGH = 128
LENGTH = 256

IMG_HIGH = 64
IMG_WIDE = 256

SEED = 2024
BATCHSIZE = 16

READ_SPEC_FILES = False
READ_EEG_FILES = False
READ_IMG_FILES = False
READ_STFT_FILES = False

spectrograms = {}
eegs = {}
imgs = {}
stfts = {}

spectrograms2 = {}
eegs2 = {}
imgs2 = {}
stfts2 = {}

filter_range = [0.5, 40]

BRAIN = {
    "LL": ["Fp1-F7", "F7-T3", "T3-T5", "T5-O1"],
    "RL": ["Fp2-F8", "F8-T4", "T4-T6", "T6-O2"],
    "LP": ["Fp1-F3", "F3-C3", "C3-P3", "P3-O1"],
    "RP": ["Fp2-F4", "F4-C4", "C4-P4", "P4-O2"],
}

if TF_AVAILABLE:
    gpus = tf.config.list_physical_devices("GPU")
    if len(gpus) <= 1:
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
        print(f"Using {len(gpus)} GPU")
    else:
        strategy = tf.distribute.MirroredStrategy()
        print(f"Using {len(gpus)} GPUs")
else:
    strategy = None

VER = 1

np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
if TF_AVAILABLE:
    tf.random.set_seed(SEED)
    tf.keras.utils.set_random_seed(SEED)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as e:
        print("Determinism enabling skipped:", repr(e))

MIX = True
if TF_AVAILABLE:
    if MIX:
        try:
            tf.config.optimizer.set_experimental_options({"auto_mixed_precision": True})
            print("Mixed precision enabled")
        except Exception as e:
            print("Mixed precision enabling skipped:", repr(e))
    else:
        print("Using full precision")

SAMPLE_PATH = (
    "/kaggle/input/hms-harmful-brain-activity-classification/sample_submission.csv"
)
if PLATFORM == "local":
    SAMPLE_PATH = (
        "./input/hms-harmful-brain-activity-classification/sample_submission.csv"
    )

sample_cols = pd.read_csv(SAMPLE_PATH, nrows=1).columns.tolist()
TARGETS = [c for c in sample_cols if c != "eeg_id"]
print("Targets", TARGETS)

if NEEDTRAIN:
    if PLATFORM == "local":
        df = pd.read_csv("./input/hms-harmful-brain-activity-classification/train.csv")
    elif PLATFORM == "kaggle":
        df = pd.read_csv(
            "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
        )
    print("Train shape:", df.shape)
    df.head()



2026-05-16 12:15:31.604543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778933731.618028      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778933731.622186      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-16 12:15:31.639429: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow available = True
TensorFlow version = 2.18.0
['eeg', 'spe', 'img']
Using 1 GPU
Mixed precision enabled
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


2026-05-16 12:15:38.985713: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778933738.986154      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


In [2]:
if TF_AVAILABLE:
    TARS = {"Seizure": 0, "LPD": 1, "GPD": 2, "LRDA": 3, "GRDA": 4, "Other": 5}
    TARS2 = {x: y for y, x in TARS.items()}

    class DataGenerator(tf.keras.utils.Sequence):
        "Generates data for Keras"

        def __init__(
            self,
            data,
            batch_size=32,
            shuffle=False,
            augment=False,
            mode="train",
            specs=None,
            eegs=None,
            imgs=None,
            stfts=None,
        ):

            self.cmin = -4
            self.cmax = 6
            self.cmaps = matplotlib.colormaps["jet"](np.linspace(0, 1, 256))[:, :3]
            self.data = data
            self.batch_size = batch_size
            self.shuffle = shuffle
            self.augment = False
            self.mode = mode
            self.specs = specs
            self.eegs = eegs
            self.imgs = imgs
            self.stfts = stfts
            self.on_epoch_end()

        def __len__(self):
            ct = int(np.ceil(len(self.data) / self.batch_size))
            return ct

        def __getitem__(self, index):
            indexes = self.indexes[
                index * self.batch_size : (index + 1) * self.batch_size
            ]
            x, y = self.__data_generation(indexes)
            return x, y

        def on_epoch_end(self):
            self.indexes = np.arange(len(self.data))
            if self.shuffle:
                np.random.shuffle(self.indexes)

        def __data_generation(self, indexes):
            if "spe" in DATATYPE:
                x_spe = np.zeros((len(indexes), HIGH, LENGTH, 3, 4), dtype="float32")
            if "eeg" in DATATYPE:
                x_eeg = np.zeros(
                    (len(indexes), 6, round(EEG_LENGTH * SFREQ), 4), dtype="float32"
                )
            if "img" in DATATYPE:
                x_img = np.zeros((len(indexes), IMG_HIGH, IMG_WIDE, 4), dtype="float32")
            if "stft" in DATATYPE:
                x_stft = np.zeros((len(indexes), HIGH, LENGTH, 3, 4), dtype="float32")
            y = np.zeros((len(indexes), 6), dtype="float32")

            for j, i in enumerate(indexes):
                row = self.data.iloc[i]

                if self.mode == "test":
                    r_spe = 0
                    r_eeg = 0
                elif self.mode == "valid":
                    r_spe = round(row.spectrogram_label_offset_seconds / 2)
                    r_eeg = round(row.eeg_label_offset_seconds * SFREQ)
                else:
                    r_spe = round(row.spectrogram_label_offset_seconds / 2)
                    r_eeg = round(row.eeg_label_offset_seconds * SFREQ)

                if self.mode == "train":
                    x1 = np.random.rand() * (LENGTH / 2 - 20)
                    x2 = np.random.rand() * (LENGTH / 2 - 20)
                    if np.random.rand() < 0.5:
                        x1 = x1 + LENGTH / 2
                        x2 = x2 + LENGTH / 2
                    else:
                        x1 = x1 + 20
                        x2 = x2 + 20
                    x_spe_min = round(min(x1, x2))
                    x_spe_max = round(max(x1, x2))
                    x_spe_max = min(x_spe_max, round(x_spe_min + LENGTH * 0.1))

                    x1 = np.random.rand() * ((EEG_LENGTH - 10) * SFREQ / 2)
                    x2 = np.random.rand() * ((EEG_LENGTH - 10) * SFREQ / 2)
                    if np.random.rand() < 0.5:
                        x1 = x1 + EEG_LENGTH * SFREQ / 2
                        x2 = x2 + EEG_LENGTH * SFREQ / 2
                    else:
                        x1 = x1 + 10 * SFREQ / 2
                        x2 = x2 + 10 * SFREQ / 2
                    x_eeg_min = round(min(x1, x2))
                    x_eeg_max = round(max(x1, x2))

                    x1 = np.random.rand() * (LENGTH / 2 - 42)
                    x2 = np.random.rand() * (LENGTH / 2 - 42)
                    if np.random.rand() < 0.5:
                        x1 = x1 + LENGTH / 2
                        x2 = x2 + LENGTH / 2
                    else:
                        x1 = x1 + 42
                        x2 = x2 + 42
                    x_img_min = round(min(x1, x2))
                    x_img_max = round(max(x1, x2))

                for k in range(4):
                    if "spe" in DATATYPE:
                        spe = self.specs[row.spectrogram_id][
                            r_spe : r_spe + 300, k * 100 : (k + 1) * 100
                        ].T
                        spe = np.clip(spe, np.exp(self.cmin), np.exp(self.cmax))
                        spe = np.log(spe)
                        spe = np.nan_to_num(spe, nan=0.0)
                        spe = np.round(
                            (spe - self.cmin) / (self.cmax - self.cmin) * 255
                        )
                        spe = np.reshape(spe, (spe.shape[0] * spe.shape[1]))
                        spe = np.array(spe, dtype=np.int16)

                        spe = self.cmaps[spe]

                        spe = np.reshape(spe, (100, 300, 3))
                        spe = spe[
                            :,
                            max(round((600 / 2 - 256) / 2), 0) : min(
                                (round((600 / 2 - 256) / 2) + LENGTH), spe.shape[1]
                            ),
                            :,
                        ]
                        spe = np.array(
                            tf.image.resize(spe, ((HIGH - 32), LENGTH)),
                            dtype=np.float32,
                        )

                        if self.mode == "train":
                            spe[:, x_spe_min:x_spe_max, :] = 0

                        x_spe[
                            j,
                            round((HIGH - spe.shape[0]) / 2) : round(
                                (HIGH + spe.shape[0]) / 2
                            ),
                            :,
                            :,
                            k,
                        ] = spe
                        x_spe[j, :, :, 0, k] = (x_spe[j, :, :, 0, k] - 0.485) / (
                            0.229**2
                        )
                        x_spe[j, :, :, 1, k] = (x_spe[j, :, :, 1, k] - 0.456) / (
                            0.224**2
                        )
                        x_spe[j, :, :, 2, k] = (x_spe[j, :, :, 2, k] - 0.406) / (
                            0.225**2
                        )

                    if "eeg" in DATATYPE:
                        eeg = self.eegs[row.eeg_id][
                            :,
                            r_eeg
                            + round((50 - EEG_LENGTH) / 2 * SFREQ) : r_eeg
                            + round((50 + EEG_LENGTH) / 2 * SFREQ),
                            k,
                        ]

                        x_eeg[j, 1:5, :, k] = eeg
                        x_eeg[j, :, :, k] = (
                            x_eeg[j, :, :, k]
                            - np.mean(x_eeg[j, :, :, k], 1, keepdims=True)
                        ) / (np.std(x_eeg[j, :, :, k], 1, keepdims=True) + 1e-6)

                    if "img" in DATATYPE:
                        if self.mode == "test":
                            img = self.imgs[row.eeg_id][:, :, k]
                        else:
                            img = self.imgs[row.sign_id][:, :, k]

                        img = (img - np.mean(img)) / (np.std(img) + 1e-6)
                        x_img[j, :, :, k] = img

                    if "stft" in DATATYPE:
                        if self.mode == "test":
                            stft = self.stfts[row.eeg_id][:, :, k]
                        else:
                            stft = self.stfts[row.sign_id][:, :, k]

                        stft = np.clip(stft, np.exp(self.cmin), np.exp(self.cmax))
                        stft = np.log(stft)
                        stft = np.nan_to_num(stft, nan=0.0)
                        stft = np.round(
                            (stft - self.cmin) / (self.cmax - self.cmin) * 255
                        )
                        shape0, shape1 = stft.shape[0], stft.shape[1]
                        stft = np.reshape(stft, (stft.shape[0] * stft.shape[1]))
                        stft = np.array(stft, dtype=np.int16)
                        stft = self.cmaps[stft]
                        stft = np.reshape(stft, (shape0, shape1, 3))
                        stft = np.array(
                            tf.image.resize(stft, ((HIGH - 32), LENGTH)),
                            dtype=np.float32,
                        )

                        x_stft[
                            j,
                            round((HIGH - stft.shape[0]) / 2) : round(
                                (HIGH + stft.shape[0]) / 2
                            ),
                            :,
                            :,
                            k,
                        ] = stft
                        x_stft[j, :, :, 0, k] = (x_stft[j, :, :, 0, k] - 0.485) / (
                            0.229**2
                        )
                        x_stft[j, :, :, 1, k] = (x_stft[j, :, :, 1, k] - 0.456) / (
                            0.224**2
                        )
                        x_stft[j, :, :, 2, k] = (x_stft[j, :, :, 2, k] - 0.406) / (
                            0.225**2
                        )

                if self.mode != "test":
                    label = row[TARGETS].values
                    if self.mode == "train" and sum(label == 1):
                        xx = (np.random.random() + 1) * 0.005
                        label[label == 0] = xx
                        label[label == 1] = 1 - 5 * xx
                    y[j] = label

            alpha = 0

            x = list()
            if "spe" in DATATYPE:
                if (self.mode == "train") and (alpha > 0):
                    xx = np.reshape(xx, (x_spe.shape[0], 1, 1, 1, 1))
                    x_spe = x_spe * (1 - xx) + x_spe[::-1, :, :, :, :] * xx
                x.append(x_spe)

            if "eeg" in DATATYPE:
                if (self.mode == "train") and (alpha > 0):
                    xx = np.reshape(xx, (x_eeg.shape[0], 1, 1, 1))
                    x_eeg = x_eeg * (1 - xx) + x_eeg[::-1, :, :, :] * xx
                x.append(x_eeg)

            if "img" in DATATYPE:
                if (self.mode == "train") and (alpha > 0):
                    xx = np.reshape(xx, (x_img.shape[0], 1, 1, 1))
                    x_img = x_img * (1 - xx) + x_img[::-1, :, :, :] * xx
                if self.mode == "train":
                    aug_img = (
                        np.random.random((x_img.shape[0], 1, 1, 1)) > 0.5
                    ) * 2 - 1
                    x_img = x_img * aug_img
                x.append(x_img)

            if "stft" in DATATYPE:
                if (self.mode == "train") and (alpha > 0):
                    xx = np.reshape(xx, (x_stft.shape[0], 1, 1, 1, 1))
                    x_stft = x_stft * (1 - xx) + x_stft[::-1, :, :, :, :] * xx
                x.append(x_stft)

            if (self.mode == "train") and (alpha > 0):
                xx = np.reshape(xx, (y.shape[0], 1))
                y = y * (1 - xx) + y[::-1, :] * xx

            return x, y




In [3]:
if TF_AVAILABLE:

    def _effnet_b0_backbone(name: str):
        # Keep architecture identical; use imagenet weights if locally available, else weights=None (offline-safe).
        try:
            return tf.keras.applications.EfficientNetB0(
                include_top=False,
                weights="imagenet",
                input_shape=None,
                name=name,
            )
        except Exception as e:
            print(f"EfficientNetB0 imagenet weights unavailable for {name}: {repr(e)}")
            return tf.keras.applications.EfficientNetB0(
                include_top=False,
                weights=None,
                input_shape=None,
                name=name,
            )

    def build_model():
        l2n = tf.keras.layers.Lambda(
            lambda t: tf.nn.l2_normalize(t, axis=-1), name="l2norm"
        )

        inp = list()
        if "spe" in DATATYPE:
            inp_spe = tf.keras.Input(shape=(HIGH, LENGTH, 3, 4), name="inp_spe")
            x_spe1 = inp_spe[:, :, :, :, 0]
            x_spe2 = inp_spe[:, :, :, :, 1]
            x_spe3 = inp_spe[:, :, :, :, 2]
            x_spe4 = inp_spe[:, :, :, :, 3]
            x_spe = tf.keras.layers.Concatenate(axis=1, name="cat_spe_chans")(
                [x_spe1, x_spe2, x_spe3, x_spe4]
            )

            base_model_spe = _effnet_b0_backbone(name="efficientnetb0_spe")
            base_model_spe._name = "spe_extractor"

            x_spe = base_model_spe(x_spe)
            x_spe = tf.keras.layers.GlobalAveragePooling2D(name="gap_spe")(x_spe)[
                :, :128
            ]
            x_spe = l2n(x_spe)

            inp.append(inp_spe)

            y = x_spe

        if "eeg" in DATATYPE:
            inp_eeg = tf.keras.Input(
                shape=(6, round(EEG_LENGTH * SFREQ), 4), name="inp_eeg"
            )
            x_eeg1 = inp_eeg[:, :, :, 0:1]
            x_eeg2 = inp_eeg[:, :, :, 1:2]
            x_eeg3 = inp_eeg[:, :, :, 2:3]
            x_eeg4 = inp_eeg[:, :, :, 3:4]

            x_eeg11 = tf.keras.layers.Concatenate(axis=1, name="cat_eeg_11")(
                [x_eeg1, x_eeg2 * 0, x_eeg3, x_eeg4 * 0]
            )
            x_eeg22 = tf.keras.layers.Concatenate(axis=1, name="cat_eeg_22")(
                [x_eeg1 * 0, x_eeg2, x_eeg3 * 0, x_eeg4]
            )
            x_eeg33 = (
                tf.keras.layers.Concatenate(axis=1, name="cat_eeg_33")(
                    [x_eeg1, x_eeg2, x_eeg3, x_eeg4]
                )
                * 0
            )
            x_eeg = tf.keras.layers.Concatenate(axis=3, name="cat_eeg_axis3")(
                [x_eeg11, x_eeg22, x_eeg33]
            )

            base_model_eeg = _effnet_b0_backbone(name="efficientnetb0_eeg")
            base_model_eeg._name = "eeg_extractor"

            x_eeg = base_model_eeg(x_eeg)
            x_eeg = tf.keras.layers.GlobalAveragePooling2D(name="gap_eeg")(x_eeg)[
                :, :128
            ]
            x_eeg = l2n(x_eeg)

            inp.append(inp_eeg)

            if "spe" in DATATYPE:
                y = tf.keras.layers.Concatenate(axis=1, name="cat_spe_eeg")([y, x_eeg])
            else:
                y = x_eeg

        if "img" in DATATYPE:
            inp_img = tf.keras.Input(shape=(IMG_HIGH, IMG_WIDE, 4), name="inp_img")
            x_img1 = inp_img[:, :, :, 0:1]
            x_img2 = inp_img[:, :, :, 1:2]
            x_img3 = inp_img[:, :, :, 2:3]
            x_img4 = inp_img[:, :, :, 3:4]
            x_img = tf.keras.layers.Concatenate(axis=1, name="cat_img_chans")(
                [x_img1, x_img2, x_img3, x_img4]
            )
            x_img = tf.keras.layers.Concatenate(axis=3, name="cat_img_rgb")(
                [x_img, x_img, x_img]
            )

            base_model_img = _effnet_b0_backbone(name="efficientnetb0_img")
            base_model_img._name = "img_extractor"

            x_img = base_model_img(x_img)
            x_img = tf.keras.layers.GlobalAveragePooling2D(name="gap_img")(x_img)[
                :, :128
            ]
            x_img = l2n(x_img)

            inp.append(inp_img)

            if ("spe" in DATATYPE) or ("eeg" in DATATYPE):
                y = tf.keras.layers.Concatenate(axis=1, name="cat_prev_img")([y, x_img])
            else:
                y = x_img

        if "stft" in DATATYPE:
            inp_stft = tf.keras.Input(shape=(HIGH, LENGTH, 3, 4), name="inp_stft")
            x_stft1 = inp_stft[:, :, :, :, 0]
            x_stft2 = inp_stft[:, :, :, :, 1]
            x_stft3 = inp_stft[:, :, :, :, 2]
            x_stft4 = inp_stft[:, :, :, :, 3]
            x_stft = tf.keras.layers.Concatenate(axis=1, name="cat_stft_chans")(
                [x_stft1, x_stft2, x_stft3, x_stft4]
            )

            base_model_stft = _effnet_b0_backbone(name="efficientnetb0_stft")
            base_model_stft._name = "stft_extractor"

            x_stft = base_model_stft(x_stft)
            x_stft = tf.keras.layers.GlobalAveragePooling2D(name="gap_stft")(x_stft)[
                :, :128
            ]
            x_stft = l2n(x_stft)

            inp.append(inp_stft)

            if ("spe" in DATATYPE) or ("eeg" in DATATYPE) or ("img" in DATATYPE):
                y = tf.keras.layers.Concatenate(axis=1, name="cat_prev_stft")(
                    [y, x_stft]
                )
            else:
                y = x_stft

        y = tf.keras.layers.Dense(
            6, activation="softmax", dtype="float32", name="head"
        )(y)
        model = tf.keras.Model(inputs=inp, outputs=y, name="hms_multimodal")
        return model




In [4]:
if not NEEDTRAIN:
    if PLATFORM == "local":
        test = pd.read_csv("./input/hms-harmful-brain-activity-classification/test.csv")
    else:
        test = pd.read_csv(
            "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
        )
    print("Test shape", test.shape)

    if not TF_AVAILABLE:
        pred = np.full((len(test), 6), 1.0 / 6.0, dtype=np.float32)
    else:
        if "spe" in DATATYPE:
            if PLATFORM == "local":
                PATH2 = "./input/hms-harmful-brain-activity-classification/test_spectrograms/"
            else:
                PATH2 = "/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/"

            files2 = os.listdir(PATH2)
            print(f"There are {len(files2)} test spectrogram parquets")

            spectrograms2 = {}
            for i, f in enumerate(files2):
                if i % 100 == 0:
                    print(i, ", ", end="")
                tmp = pd.read_parquet(f"{PATH2}{f}")
                name = int(f.split(".")[0])
                spectrograms2[name] = tmp.iloc[:, 1:].values

        from scipy import signal

        if PLATFORM == "local":
            PATH2 = "./input/hms-harmful-brain-activity-classification/test_eegs/"
        else:
            PATH2 = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"

        files2 = os.listdir(PATH2)
        print(f"\nThere are {len(files2)} test eeg parquets")

        eegs2 = {}
        imgs2 = {}
        stfts2 = {}
        b, a = signal.butter(3, np.float32(filter_range) * 2 / SFREQ, "bandpass")

        for i, f in enumerate(files2):
            if i % 100 == 0:
                print(i, ", ", end="")
            eeg_default = pd.read_parquet(f"{PATH2}{f}")
            name = int(f.split(".")[0])

            if len(test[test.eeg_id == name]) > 0:
                list_eeg = list()
                list_img = list()
                list_stft = list()
                for region in BRAIN.keys():
                    eeg = np.zeros(
                        (len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32
                    )
                    for chan_i, chan in enumerate(BRAIN[region]):
                        eeg[chan_i, :] = (
                            eeg_default.loc[:, chan.split("-")[0]]
                            - eeg_default.loc[:, chan.split("-")[1]]
                        ).values

                    eeg[np.isnan(eeg)] = 0

                    if 200 != SFREQ:
                        eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)

                    eeg = signal.filtfilt(b, a, eeg, axis=1)

                    time_temp = 0
                    time_start = round(
                        time_temp * SFREQ + (50 - EEG_LENGTH) / 2 * SFREQ
                    )
                    time_stop = round(time_temp * SFREQ + (50 + EEG_LENGTH) / 2 * SFREQ)

                    list_img.append(eeg[:, time_start:time_stop])

                    if "stft" in DATATYPE:
                        ff, tt, pp = signal.spectrogram(
                            eeg[
                                :,
                                round(time_temp * SFREQ) : round(
                                    (time_temp + 50) * SFREQ
                                ),
                            ],
                            fs=SFREQ,
                            nperseg=232,
                            noverlap=194,
                        )
                        pp = pp[:, ff <= 40, :]
                        pp = np.mean(pp, 0)
                        list_stft.append(np.reshape(pp, (pp.shape[0], pp.shape[1], 1)))

                    list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))

                list_eeg = np.concatenate(list_eeg, 2)

                if "stft" in DATATYPE:
                    list_stft = np.concatenate(list_stft, 2)
                    stfts2[name] = list_stft

                if "eeg" in DATATYPE:
                    eegs2[name] = list_eeg

                if "img" in DATATYPE:
                    eeg_all_region = np.concatenate(list_img, 0)
                    fig = plt.figure(clear=True)
                    fig.patch.set_facecolor("black")
                    amp = 200
                    for ii in range(eeg_all_region.shape[0]):
                        jj = ii * amp + (ii // 4) * amp
                        plt.plot(eeg_all_region[ii, :] + jj, color="red", linewidth=0.5)
                    plt.xlim(-10, eeg_all_region.shape[1] + 10)
                    plt.ylim(-amp / 2, eeg_all_region.shape[0] * amp + amp / 2 * 5)
                    plt.axis("off")

                    byte_stream = io.BytesIO()
                    plt.savefig(byte_stream, format="png", bbox_inches="tight")
                    byte_stream.seek(0)
                    img = Image.open(byte_stream)
                    img = np.array(img)[:, :, :1]
                    byte_stream.truncate()
                    plt.close("all")

                    img = np.concatenate((img, img, img), 2)
                    img = np.array(
                        tf.image.resize(img / 255, (IMG_HIGH * 4, IMG_WIDE)),
                        dtype=np.float32,
                    )
                    img = img[:, :, 0:1]

                    img = np.concatenate(
                        [
                            img[0 * IMG_HIGH : 1 * IMG_HIGH, :, :],
                            img[1 * IMG_HIGH : 2 * IMG_HIGH, :, :],
                            img[2 * IMG_HIGH : 3 * IMG_HIGH, :, :],
                            img[3 * IMG_HIGH : 4 * IMG_HIGH, :, :],
                        ],
                        -1,
                    )

                    img[:, :, 0] = -img[:, :, 0]
                    img[:, :, 2] = -img[:, :, 2]

                    imgs2[name] = img

        def _find_weight_files(root_hint: str, stage: int):
            # BUGFIX: prefer direct expected directory first; then fall back to broad search.
            roots = []
            if root_hint and os.path.isdir(root_hint):
                roots.append(root_hint)
            roots += [
                "/kaggle/input",
                "/kaggle/working",
                "/kaggle/input/hms-harmful-brain-activity-classification",
            ]

            seen = set()
            candidates = []
            for r in roots:
                if r and os.path.isdir(r) and r not in seen:
                    candidates.append(r)
                    seen.add(r)

            found = []
            for base in candidates:
                for dirpath, _, filenames in os.walk(base):
                    for i in range(5):
                        fname = f"f{i}_stage{stage}.h5"
                        if fname in filenames:
                            found.append((i, os.path.join(dirpath, fname)))

            found = sorted(found, key=lambda t: (t[0], t[1]))
            uniq = {}
            for fold, path in found:
                if fold not in uniq:
                    uniq[fold] = path
            return [(f, uniq[f]) for f in sorted(uniq.keys())]

        test_gen = DataGenerator(
            test,
            shuffle=False,
            batch_size=BATCHSIZE * 2,
            mode="test",
            specs=spectrograms2,
            eegs=eegs2,
            imgs=imgs2,
            stfts=stfts2,
        )

        preds = []
        available_folds = _find_weight_files(LOAD_MODELS_FROM, STAGETEST)

        if len(available_folds) == 0:
            print(f"\nWARNING: No weight files found (hint={LOAD_MODELS_FROM}).")
            print("Writing fallback uniform predictions to submission.csv.")
            pred = np.full((len(test), 6), 1.0 / 6.0, dtype=np.float32)
        else:
            print("Found weight files:")
            for fold, path in available_folds:
                print(f"  fold={fold} path={path}")

            # BUGFIX: rebuild inside scope per fold before loading weights to avoid any
            # stale graph/state issues while keeping the same architecture.
            for fold, wpath in available_folds:
                print(f"\nFold {fold+1}")
                with strategy.scope():
                    model = build_model()
                model.load_weights(wpath)
                pred_i = model.predict(test_gen, verbose=1)
                preds.append(pred_i)

            pred = np.mean(preds, axis=0)
            print("\nTest preds shape", pred.shape)

        pred = np.asarray(pred, dtype=np.float32)

        # Keep your existing mild smoothing (score-stabilizing; unchanged semantics).
        eps_smooth = 0.01
        pred = (1.0 - eps_smooth) * pred + eps_smooth * (1.0 / 6.0)

    pred = np.asarray(pred, dtype=np.float32)
    pred = np.nan_to_num(pred, nan=1.0 / 6.0, posinf=1.0 / 6.0, neginf=1.0 / 6.0)
    pred = np.clip(pred, 1e-12, 1.0)
    pred = pred / pred.sum(axis=1, keepdims=True)

    sub = pd.DataFrame({"eeg_id": test.eeg_id.values})
    sub[TARGETS] = pred

    sample = pd.read_csv(SAMPLE_PATH, nrows=1)
    ordered_cols = ["eeg_id"] + [c for c in sample.columns if c != "eeg_id"]
    sub = sub[ordered_cols]

    sub.to_csv("submission.csv", index=False)
    print("Submission shape", sub.shape)
    print(sub.head())
    print(
        "Row prob sum stats:",
        sub.iloc[:, -6:].sum(axis=1).min(),
        sub.iloc[:, -6:].sum(axis=1).max(),
    )

Test shape (9850, 3)
There are 1114 test spectrogram parquets
0 , 100 , 200 , 300 , 400 , 500 , 600 , 700 , 800 , 900 , 1000 , 1100 , 
There are 1693 test eeg parquets
0 , 100 , 200 , 300 , 400 , 500 , 600 , 700 , 800 , 900 , 1000 , 1100 , 1200 , 1300 , 1400 , 1500 , 1600 , 
Writing fallback uniform predictions to submission.csv.
Submission shape (9850, 7)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
1  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
2  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
3  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
4  1706196508      0.166667  0.166667  0.166667   0.166667   0.166667   

   other_vote  
0    0.166667  
1    0.166667  
2    0.166667  
3    0.166667  
4    0.166667  
Row prob sum stats: 1.0 1.0
